# Step 9: Subject-Wise Train / Validation / Test Split

Split the combined EEG dataset at the SUBJECT level to prevent data leakage.

Rules:
- No model training, no feature extraction, no augmentation
- Split based on subject_key (dataset + subject_id)
- 70% train, 15% validation, 15% test
- Stratified by diagnosis (AD, FTD, HC)
- MCI and UNKNOWN excluded from primary split but kept in metadata
- Fixed random seed: 42

In [10]:
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" "scikit-learn==1.2.2" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.
conda-repo-cli 1.0.20 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.20 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.
botocore 1.27.28 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
COMBINED_DIR = PROCESSED_DIR / 'combined'
SPLITS_DIR = PROCESSED_DIR / 'splits'
COMBINED_META_PATH = COMBINED_DIR / 'combined_metadata.csv'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
TRAIN_SIZE = 0.70

print(f'Split ratios: Train={TRAIN_SIZE}, Val={VAL_SIZE}, Test={TEST_SIZE}')
print(f'Random state: {RANDOM_STATE}')

Split ratios: Train=0.7, Val=0.15, Test=0.15
Random state: 42


## 1. Load Combined Metadata and Create Subject-Level Table

In [12]:
meta = pd.read_csv(COMBINED_META_PATH)
print(f'Loaded {len(meta)} segments')
print(f'Datasets: {meta["dataset"].unique()}')
print(f'Labels: {meta["standard_label"].unique()}')

# Create subject_key
meta['subject_key'] = meta['dataset'] + '_' + meta['subject_id']

# Subject-level table (one row per subject)
subject_df = meta.groupby('subject_key').agg(
    dataset=('dataset', 'first'),
    subject_id=('subject_id', 'first'),
    standard_label=('standard_label', 'first')
).reset_index()

print(f'Total unique subjects: {len(subject_df)}')
print(f'Labels:')
print(subject_df['standard_label'].value_counts())
print(f'\nBy dataset:')
print(subject_df.groupby('dataset')['standard_label'].value_counts())

# Verify each subject has only one diagnosis
label_counts = meta.groupby('subject_key')['standard_label'].nunique()
multi_label = label_counts[label_counts > 1]
if len(multi_label) > 0:
    print(f'WARNING: {len(multi_label)} subjects have multiple labels:')
    for s in multi_label.index:
        print(f'  {s}: {meta[meta["subject_key"]==s]["standard_label"].unique()}')
else:
    print('All subjects have single diagnosis ✓')

# Check for duplicate subject IDs across datasets
dup_subj_id = meta.groupby('subject_id')['dataset'].nunique()
dup_subj_id = dup_subj_id[dup_subj_id > 1]
if len(dup_subj_id) > 0:
    print(f'Subject IDs appearing in multiple datasets:')
    for s in dup_subj_id.index:
        print(f'  {s}: {meta[meta["subject_id"]==s]["dataset"].unique()}')
else:
    print('No duplicate subject IDs across datasets ✓')

Loaded 6381 segments
Datasets: ['ADFTD' 'AD_Auditory']
Labels: ['AD' 'HC' 'FTD' 'UNKNOWN' 'MCI']
Total unique subjects: 88
Labels:
AD         36
HC         27
FTD        17
MCI         6
UNKNOWN     2
Name: standard_label, dtype: int64

By dataset:
dataset      standard_label
ADFTD        AD                23
             FTD               17
             HC                17
AD_Auditory  AD                13
             HC                10
             MCI                6
             UNKNOWN            2
Name: standard_label, dtype: int64
All subjects have single diagnosis ✓
No duplicate subject IDs across datasets ✓


## 2. Prepare for Stratified Split (AD, FTD, HC only)

In [13]:
# Primary classification labels
PRIMARY_LABELS = ['AD', 'FTD', 'HC']

# Filter subjects for primary split
primary_subjects = subject_df[subject_df['standard_label'].isin(PRIMARY_LABELS)].copy()
excluded_subjects = subject_df[~subject_df['standard_label'].isin(PRIMARY_LABELS)].copy()

print(f'Primary split subjects: {len(primary_subjects)}')
print(f'  AD: {len(primary_subjects[primary_subjects["standard_label"]=="AD"])}')
print(f'  FTD: {len(primary_subjects[primary_subjects["standard_label"]=="FTD"])}')
print(f'  HC: {len(primary_subjects[primary_subjects["standard_label"]=="HC"])}')
print(f'\nExcluded (MCI/UNKNOWN): {len(excluded_subjects)}')
print(excluded_subjects[['subject_key', 'standard_label']].to_string(index=False))

Primary split subjects: 80
  AD: 36
  FTD: 17
  HC: 27

Excluded (MCI/UNKNOWN): 8
       subject_key standard_label
AD_Auditory_sub-06        UNKNOWN
AD_Auditory_sub-13        UNKNOWN
AD_Auditory_sub-17            MCI
AD_Auditory_sub-18            MCI
AD_Auditory_sub-19            MCI
AD_Auditory_sub-27            MCI
AD_Auditory_sub-28            MCI
AD_Auditory_sub-32            MCI


## 3. Perform Stratified Subject-Level Split

In [14]:
# First split: train (70%) vs temp (30%)
train_subjects, temp_subjects = train_test_split(
    primary_subjects,
    test_size=(TEST_SIZE + VAL_SIZE),
    stratify=primary_subjects['standard_label'],
    random_state=RANDOM_STATE
)

# Second split: validation (15%) vs test (15%) from temp
val_size_adjusted = VAL_SIZE / (TEST_SIZE + VAL_SIZE)  # 0.5
val_subjects, test_subjects = train_test_split(
    temp_subjects,
    test_size=0.5,
    stratify=temp_subjects['standard_label'],
    random_state=RANDOM_STATE
)

print(f'Train subjects: {len(train_subjects)}')
print(f'Validation subjects: {len(val_subjects)}')
print(f'Test subjects: {len(test_subjects)}')
print(f'Total primary: {len(primary_subjects)}')

# Add split column
train_subjects = train_subjects.copy()
val_subjects = val_subjects.copy()
test_subjects = test_subjects.copy()
train_subjects['split'] = 'train'
val_subjects['split'] = 'validation'
test_subjects['split'] = 'test'

# Combine
split_subjects = pd.concat([train_subjects, val_subjects, test_subjects], ignore_index=True)

print('\nSubject distribution by split and label:')
print(split_subjects.groupby(['split', 'standard_label']).size().unstack(fill_value=0))

Train subjects: 56
Validation subjects: 12
Test subjects: 12
Total primary: 80

Subject distribution by split and label:
standard_label  AD  FTD  HC
split                      
test             6    2   4
train           25   12  19
validation       5    3   4


## 4. Map Split Assignment to All Segments

In [15]:
# Create subject_key -> split mapping
subject_split_map = dict(zip(split_subjects['subject_key'], split_subjects['split']))

# Map to all segments
meta['split'] = meta['subject_key'].map(subject_split_map)

# Excluded subjects get 'excluded' split
meta['split'] = meta['split'].fillna('excluded')

print('Segment split distribution:')
print(meta['split'].value_counts())
print('\nSegment split by label:')
print(meta.groupby(['split', 'standard_label']).size().unstack(fill_value=0))

Segment split distribution:
train         4505
test           969
validation     763
excluded       144
Name: split, dtype: int64

Segment split by label:
standard_label    AD   FTD    HC  MCI  UNKNOWN
split                                         
excluded           0     0     0  114       30
test             531   180   258    0        0
train           1879  1108  1518    0        0
validation       273   248   242    0        0


## 5. Leakage Verification

In [16]:
# Get subject keys per split
train_keys = set(split_subjects[split_subjects['split'] == 'train']['subject_key'])
val_keys = set(split_subjects[split_subjects['split'] == 'validation']['subject_key'])
test_keys = set(split_subjects[split_subjects['split'] == 'test']['subject_key'])

# Check overlaps
train_val_overlap = train_keys & val_keys
train_test_overlap = train_keys & test_keys
val_test_overlap = val_keys & test_keys

leakage_detected = False
if train_val_overlap:
    print(f'ERROR: Train-Val overlap: {train_val_overlap}')
    leakage_detected = True
if train_test_overlap:
    print(f'ERROR: Train-Test overlap: {train_test_overlap}')
    leakage_detected = True
if val_test_overlap:
    print(f'ERROR: Val-Test overlap: {val_test_overlap}')
    leakage_detected = True

# Verify each segment appears in exactly one split
seg_split_counts = meta.groupby('global_segment_id')['split'].nunique()
multi_split_segs = seg_split_counts[seg_split_counts > 1]
if len(multi_split_segs) > 0:
    print(f'ERROR: {len(multi_split_segs)} segments in multiple splits')
    leakage_detected = True

if leakage_detected:
    raise RuntimeError('ERROR: SUBJECT LEAKAGE DETECTED')
else:
    print('✓ No subject leakage detected')
    print('✓ All segments assigned to exactly one split')

✓ No subject leakage detected
✓ All segments assigned to exactly one split


## 6. Create Split Metadata Files

In [17]:
# Select columns for split metadata
split_cols = [
    'global_segment_id',
    'dataset',
    'subject_id',
    'subject_key',
    'standard_label',
    'segment_file',
    'start_time',
    'end_time',
    'sampling_rate',
    'channels',
    'num_samples',
    'split'
]

# Save train/validation/test metadata
for split_name in ['train', 'validation', 'test']:
    split_df = meta[meta['split'] == split_name][split_cols].copy()
    out_path = SPLITS_DIR / f'{split_name}_metadata.csv'
    split_df.to_csv(out_path, index=False)
    print(f'Saved {split_name}: {len(split_df)} segments to {out_path}')

# Save subject split assignment
subject_split_assignment = split_subjects[['subject_key', 'dataset', 'subject_id', 'standard_label', 'split']].copy()
subject_split_assignment.to_csv(SPLITS_DIR / 'subject_split_assignment.csv', index=False)
print(f'Saved subject_split_assignment.csv: {len(subject_split_assignment)} subjects')

Saved train: 4505 segments to /home/admin1/Desktop/SIH_2026_ALL/processed_data/splits/train_metadata.csv
Saved validation: 763 segments to /home/admin1/Desktop/SIH_2026_ALL/processed_data/splits/validation_metadata.csv
Saved test: 969 segments to /home/admin1/Desktop/SIH_2026_ALL/processed_data/splits/test_metadata.csv
Saved subject_split_assignment.csv: 80 subjects


## 7. Create Summary Tables

In [18]:
# TABLE 1: Split | Unique Subjects | Recordings | Segments
table1_rows = []
for split_name in ['train', 'validation', 'test']:
    split_meta = meta[meta['split'] == split_name]
    table1_rows.append({
        'Split': split_name,
        'Unique_Subjects': split_meta['subject_key'].nunique(),
        'Recordings': split_meta['source_file'].nunique(),
        'Segments': len(split_meta)
    })
table1 = pd.DataFrame(table1_rows)
print('TABLE 1: Split composition (all labels)')
print(table1.to_string(index=False))

# TABLE 2: Split | AD Subjects | FTD Subjects | HC Subjects
table2_rows = []
for split_name in ['train', 'validation', 'test']:
    split_subj = split_subjects[split_subjects['split'] == split_name]
    table2_rows.append({
        'Split': split_name,
        'AD_Subjects': len(split_subj[split_subj['standard_label'] == 'AD']),
        'FTD_Subjects': len(split_subj[split_subj['standard_label'] == 'FTD']),
        'HC_Subjects': len(split_subj[split_subj['standard_label'] == 'HC']),
    })
table2 = pd.DataFrame(table2_rows)
print('\nTABLE 2: Primary class subjects per split')
print(table2.to_string(index=False))

# TABLE 3: Split | AD Segments | FTD Segments | HC Segments
table3_rows = []
for split_name in ['train', 'validation', 'test']:
    split_meta = meta[(meta['split'] == split_name) & (meta['standard_label'].isin(PRIMARY_LABELS))]
    table3_rows.append({
        'Split': split_name,
        'AD_Segments': len(split_meta[split_meta['standard_label'] == 'AD']),
        'FTD_Segments': len(split_meta[split_meta['standard_label'] == 'FTD']),
        'HC_Segments': len(split_meta[split_meta['standard_label'] == 'HC']),
    })
table3 = pd.DataFrame(table3_rows)
print('\nTABLE 3: Primary class segments per split')
print(table3.to_string(index=False))

TABLE 1: Split composition (all labels)
     Split  Unique_Subjects  Recordings  Segments
     train               56          97      4505
validation               12          19       763
      test               12          21       969

TABLE 2: Primary class subjects per split
     Split  AD_Subjects  FTD_Subjects  HC_Subjects
     train           25            12           19
validation            5             3            4
      test            6             2            4

TABLE 3: Primary class segments per split
     Split  AD_Segments  FTD_Segments  HC_Segments
     train         1879          1108         1518
validation          273           248          242
      test          531           180          258


## 8. Class Distribution Comparison (Subjects vs Segments)

In [19]:
print('=== SUBJECT-LEVEL DISTRIBUTION ===')
subj_dist = split_subjects.groupby(['split', 'standard_label']).size().unstack(fill_value=0)
print(subj_dist)
print(f'\nSubject percentages:')
print((subj_dist.div(subj_dist.sum(axis=1), axis=0) * 100).round(1))

print('\n=== SEGMENT-LEVEL DISTRIBUTION ===')
seg_dist = meta[meta['standard_label'].isin(PRIMARY_LABELS)].groupby(['split', 'standard_label']).size().unstack(fill_value=0)
print(seg_dist)
print(f'\nSegment percentages:')
print((seg_dist.div(seg_dist.sum(axis=1), axis=0) * 100).round(1))

print('\n=== DISCREPANCY (Segment % - Subject %) ===')
subj_pct = subj_dist.div(subj_dist.sum(axis=1), axis=0) * 100
seg_pct = seg_dist.div(seg_dist.sum(axis=1), axis=0) * 100
discrepancy = seg_pct - subj_pct
print(discrepancy.round(1))

=== SUBJECT-LEVEL DISTRIBUTION ===
standard_label  AD  FTD  HC
split                      
test             6    2   4
train           25   12  19
validation       5    3   4

Subject percentages:
standard_label    AD   FTD    HC
split                           
test            50.0  16.7  33.3
train           44.6  21.4  33.9
validation      41.7  25.0  33.3

=== SEGMENT-LEVEL DISTRIBUTION ===
standard_label    AD   FTD    HC
split                           
test             531   180   258
train           1879  1108  1518
validation       273   248   242

Segment percentages:
standard_label    AD   FTD    HC
split                           
test            54.8  18.6  26.6
train           41.7  24.6  33.7
validation      35.8  32.5  31.7

=== DISCREPANCY (Segment % - Subject %) ===
standard_label   AD  FTD   HC
split                        
test            4.8  1.9 -6.7
train          -2.9  3.2 -0.2
validation     -5.9  7.5 -1.6


## 9. Final Summary Report and Save

In [20]:
# Create final summary
summary_rows = []
for split_name in ['train', 'validation', 'test']:
    split_meta = meta[meta['split'] == split_name]
    split_subj = split_subjects[split_subjects['split'] == split_name]
    
    summary_rows.append({
        'Split': split_name,
        'Unique_Subjects': len(split_subj),
        'Recordings': split_meta['source_file'].nunique(),
        'Segments': len(split_meta),
        'AD_Subjects': len(split_subj[split_subj['standard_label'] == 'AD']),
        'FTD_Subjects': len(split_subj[split_subj['standard_label'] == 'FTD']),
        'HC_Subjects': len(split_subj[split_subj['standard_label'] == 'HC']),
        'AD_Segments': len(split_meta[split_meta['standard_label'] == 'AD']),
        'FTD_Segments': len(split_meta[split_meta['standard_label'] == 'FTD']),
        'HC_Segments': len(split_meta[split_meta['standard_label'] == 'HC']),
    })

split_summary = pd.DataFrame(summary_rows)
split_summary.to_csv(SPLITS_DIR / 'split_summary.csv', index=False)
print('SPLIT SUMMARY:')
print(split_summary.to_string(index=False))

SPLIT SUMMARY:
     Split  Unique_Subjects  Recordings  Segments  AD_Subjects  FTD_Subjects  HC_Subjects  AD_Segments  FTD_Segments  HC_Segments
     train               56          97      4505           25            12           19         1879          1108         1518
validation               12          19       763            5             3            4          273           248          242
      test               12          21       969            6             2            4          531           180          258


In [21]:
print('='*60)
print('FINAL SPLIT REPORT')
print('='*60)

print(f'\nTotal subjects (primary): {len(primary_subjects)}')
print(f'  AD: {len(primary_subjects[primary_subjects["standard_label"]=="AD"])}')
print(f'  FTD: {len(primary_subjects[primary_subjects["standard_label"]=="FTD"])}')
print(f'  HC: {len(primary_subjects[primary_subjects["standard_label"]=="HC"])}')
print(f'\nExcluded subjects (MCI/UNKNOWN): {len(excluded_subjects)}')
for _, r in excluded_subjects.iterrows():
    print(f'  {r["subject_key"]}: {r["standard_label"]}')

print(f'\nSplit sizes:')
for split_name in ['train', 'validation', 'test']:
    s = split_subjects[split_subjects['split'] == split_name]
    print(f'  {split_name}: {len(s)} subjects ({len(s)/len(primary_subjects)*100:.1f}%)')

print('\nSUBJECT-WISE DATA SPLIT COMPLETE — NO SUBJECT LEAKAGE DETECTED')

FINAL SPLIT REPORT

Total subjects (primary): 80
  AD: 36
  FTD: 17
  HC: 27

Excluded subjects (MCI/UNKNOWN): 8
  AD_Auditory_sub-06: UNKNOWN
  AD_Auditory_sub-13: UNKNOWN
  AD_Auditory_sub-17: MCI
  AD_Auditory_sub-18: MCI
  AD_Auditory_sub-19: MCI
  AD_Auditory_sub-27: MCI
  AD_Auditory_sub-28: MCI
  AD_Auditory_sub-32: MCI

Split sizes:
  train: 56 subjects (70.0%)
  validation: 12 subjects (15.0%)
  test: 12 subjects (15.0%)

SUBJECT-WISE DATA SPLIT COMPLETE — NO SUBJECT LEAKAGE DETECTED
